# NUTDTS 816 Time Series Analysis
## L08 Forecasting with ARIMA, benchmarks, automatic selection

Lab notebook for Chapter 4 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### Carried forward from Lab 7 (run these cells first; they define the objects used below)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
import tsdata
usc = tsdata.uschange()['Consumption']; usc.name = 'consumption growth (%)'
fig, axes = plt.subplots(1, 3, figsize=(11, 3), gridspec_kw={'width_ratios': [2, 1, 1]})
usc.plot(ax=axes[0], lw=0.9, title='Quarterly US consumption growth (%)'); axes[0].set_xlabel('')
plot_acf(usc, lags=24, ax=axes[1], title='ACF'); plot_pacf(usc, lags=24, ax=axes[2], title='PACF')
for ax in axes[1:]: ax.set_ylim(-0.5, 1)
_caption = 'A stationary series. The ACF has three significant early lags and tails off; the PACF has significant lags 1 to 3. Candidates: AR(3), MA(3), or a mixed model.'

In [ ]:
cands = [(3, 0, 0), (0, 0, 3), (1, 0, 1), (1, 0, 2), (2, 0, 1), (1, 0, 3), (2, 0, 2)]
rows = []
for o in cands:
    f = ARIMA(usc, order=o).fit()
    lb = acorr_ljungbox(f.resid, lags=[8], model_df=o[0] + o[2], return_df=True)
    rows.append({'order': o, 'AIC': round(f.aic, 2), 'AICc': round(f.aicc, 2), 'BIC': round(f.bic, 2),
                 'LB(8) p': round(lb.lb_pvalue.iloc[0], 3), 'sigma2': round(f.params['sigma2'], 4)})
print(pd.DataFrame(rows).sort_values('AICc').to_string(index=False))

In [ ]:
fit = ARIMA(usc, order=(3, 0, 0)).fit()
print(fit.summary().tables[1])
fig = fit.plot_diagnostics(figsize=(10, 6))
_caption = 'statsmodels diagnostic panel for the AR(3): standardised residuals, their histogram against N(0,1), Q-Q plot, and residual correlogram.'

In [ ]:
lb = acorr_ljungbox(fit.resid, lags=[4, 8, 12, 16], model_df=3, return_df=True)
print(lb.round(3).to_string())
print('\nAR root modulus:', np.round(np.abs(fit.arroots), 3), '(all should exceed 1 for stationarity)')

## Forecasting with ARIMA, benchmarks, automatic selection

### 4.6 Forecasting with ARIMA

In [ ]:
fc = fit.get_forecast(steps=12)
mean = fc.predicted_mean; ci80 = fc.conf_int(alpha=0.2); ci95 = fc.conf_int(alpha=0.05)
ax = usc['2005':].plot(figsize=(9, 3.4), lw=1, label='observed')
mean.plot(ax=ax, color='#B8860B', lw=2, label='AR(3) forecast')
ax.fill_between(ci95.index, ci95.iloc[:, 0], ci95.iloc[:, 1], color='#B8860B', alpha=0.15, label='95% PI')
ax.fill_between(ci80.index, ci80.iloc[:, 0], ci80.iloc[:, 1], color='#B8860B', alpha=0.3, label='80% PI')
ax.axhline(fit.params['const'], color='#555555', lw=0.8, ls='--', label='estimated mean'); ax.legend(ncol=3, fontsize=8); ax.set_xlabel('')
ax.set_title('Twelve-quarter forecast of US consumption growth')
print(pd.DataFrame({'forecast': mean, 'se': fc.se_mean}).round(3).head(6).to_string())
_caption = 'Forecasts converge to the mean within a few quarters; the interval width stabilises at the unconditional spread of the series.'

### 4.7 Worked example II: an integrated series (the exchange rate)

In [ ]:
fx = tsdata.nigeria_fx(); y = np.log(fx)
train, test = y[:'2025-06'], y['2025-07':]
d1 = train.diff().dropna()
fig, axes = plt.subplots(1, 3, figsize=(11, 3), gridspec_kw={'width_ratios': [2, 1, 1]})
d1.plot(ax=axes[0], lw=0.9, title='∇ log NGN/USD (training period)'); axes[0].set_xlabel('')
plot_acf(d1, lags=24, ax=axes[1], title='ACF'); plot_pacf(d1, lags=24, ax=axes[2], title='PACF')
for ax in axes[1:]: ax.set_ylim(-0.5, 1)
print(f'mean monthly log change = {d1.mean():.4f}  (≈ {100*d1.mean():.2f}% per month; drift is clearly non-zero)')
_caption = 'The differenced series is dominated by a few devaluation spikes; apart from them it is close to white noise with a positive mean.'

In [ ]:
rows = []
for o in [(0, 1, 0), (1, 1, 0), (0, 1, 1), (1, 1, 1), (2, 1, 0)]:
    f = ARIMA(train, order=o, trend='t' if o[1] == 1 else 'c').fit()   # trend='t' = drift when d=1
    lb = acorr_ljungbox(f.resid[1:], lags=[10], model_df=o[0] + o[2], return_df=True)
    rows.append({'order': o, 'AICc': round(f.aicc, 1), 'BIC': round(f.bic, 1), 'LB(10) p': round(lb.lb_pvalue.iloc[0], 3)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
m_rw = ARIMA(train, order=(0, 1, 0), trend='t').fit()
m_ar = ARIMA(train, order=(1, 1, 0), trend='t').fit()
h = len(test)
def fc_frame(m, h): 
    f = m.get_forecast(h); ci = f.conf_int(alpha=0.2)
    return pd.DataFrame({'mean': f.predicted_mean, 'lo80': ci.iloc[:, 0], 'hi80': ci.iloc[:, 1]})
f_rw, f_ar = fc_frame(m_rw, h), fc_frame(m_ar, h)
ax = np.exp(y['2022':]).plot(figsize=(9, 3.4), lw=1, label='observed (incl. hold-out)')
np.exp(f_rw['mean']).plot(ax=ax, color='#B8860B', lw=2, label='ARIMA(0,1,0)+drift')
ax.fill_between(f_rw.index, np.exp(f_rw.lo80), np.exp(f_rw.hi80), color='#B8860B', alpha=0.2, label='80% PI')
np.exp(f_ar['mean']).plot(ax=ax, color='#2F6DB5', lw=1.5, ls='--', label='ARIMA(1,1,0)+drift')
ax.axvline(test.index[0], color='#555555', lw=0.8); ax.legend(fontsize=8); ax.set_xlabel(''); ax.set_title('NGN/USD: 12-month forecasts from June 2025 (simulated data)')
_caption = 'Random walk with drift: a straight line in logs (a constant percentage depreciation) with an interval that widens with the square root of the horizon.'

In [ ]:
se = m_rw.get_forecast(h).se_mean
median_fc = np.exp(f_rw['mean']); mean_fc = np.exp(f_rw['mean'] + 0.5 * se**2)
print(pd.DataFrame({'median forecast': median_fc, 'mean forecast': mean_fc, 'ratio': mean_fc / median_fc}).round(3).iloc[[0, 5, 11]].to_string())

### 4.8 Benchmarks: the methods every model must beat

In [ ]:
def naive(train, h):  return pd.Series(np.repeat(train.iloc[-1], h), index=pd.date_range(train.index[-1], periods=h+1, freq='MS')[1:])
def drift(train, h):
    slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
    return naive(train, h) + slope * np.arange(1, h + 1)
def mean_fc(train, h): return naive(train, h) * 0 + train.mean()

actual = np.exp(test)
results = {'Mean': np.exp(mean_fc(train, h)), 'Naive': np.exp(naive(train, h)), 'Drift': np.exp(drift(train, h)),
           'ARIMA(0,1,0)+drift': np.exp(f_rw['mean']), 'ARIMA(1,1,0)+drift': np.exp(f_ar['mean'])}
tab = pd.DataFrame({k: {'MAE': (v - actual).abs().mean(), 'RMSE': np.sqrt(((v - actual)**2).mean())} for k, v in results.items()}).T
print(tab.round(1).to_string())

### 4.9 Automatic order selection

In [ ]:
import pmdarima as pm
auto = pm.auto_arima(usc, seasonal=False, stepwise=True, information_criterion='aicc', suppress_warnings=True, trace=False)
print('pmdarima choice for consumption growth:', auto.order, ' AICc =', round(auto.aicc(), 2))
auto_fx = pm.auto_arima(train, seasonal=False, stepwise=True, information_criterion='aicc', suppress_warnings=True)
print('pmdarima choice for log NGN/USD:', auto_fx.order, ' with trend:', auto_fx.with_intercept)

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, Naive, RandomWalkWithDrift, SeasonalNaive
df = pd.DataFrame({'unique_id': 'usc', 'ds': usc.index, 'y': usc.values})
sf = StatsForecast(models=[AutoARIMA(season_length=1), Naive(), RandomWalkWithDrift()], freq='QS')
sf.fit(df)
print('statsforecast AutoARIMA:', sf.fitted_[0, 0].model_['arma'], ' (p, q, P, Q, m, d, D)')
print(sf.predict(h=4).round(3).to_string(index=False))

## Exercises

4. Take the `goog` series (daily closing prices). Show that ARIMA(0,1,0) is hard to improve on by AICc, compute the half-life of a shock in the best AR(1) on the differences, and explain in two sentences why the 95% interval at horizon 30 is so wide.
5. Explain why AIC cannot be used to decide between ARIMA(1,0,1) and ARIMA(1,1,1) for the same series, and what should be used instead.
6. A model has residual Ljung-Box p-value 0.001 at lag 12 with a single significant residual autocorrelation at lag 12. What is missing, and what would you try next?

In [ ]:
# Your work here
